# Hybrid Recommendations

## 1. Data Preparation

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os, zipfile, subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# Download dataset if needed
zip_path = "amazon-product-reviews.zip"
if not os.path.exists(zip_path):
    print("Downloading dataset...")
    subprocess.run(["kaggle", "datasets", "download", "-d", "arhamrumi/amazon-product-reviews"], check=True)
    print("Download complete.")

# Extract if needed
csv_path = "Reviews.csv"
if not os.path.exists(csv_path):
    print("Extracting zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")
    print("Extraction complete.")

# Load and clean
print("Loading Reviews.csv...")
df = pd.read_csv("Reviews.csv")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

df["ProfileName"] = df["ProfileName"].fillna("Unknown")
df["Summary"]     = df["Summary"].fillna("")
df["full_text"]   = (df["Summary"] + " " + df["Text"]).str.strip()
df["review_time"] = pd.to_datetime(df["Time"], unit="s")
df["helpfulness_ratio"] = np.where(
    df["HelpfulnessDenominator"] > 0,
    df["HelpfulnessNumerator"] / df["HelpfulnessDenominator"], 0
)
df["text_length_chars"] = df["Text"].astype(str).apply(len)
df["text_length_words"] = df["Text"].astype(str).apply(lambda x: len(x.split()))
print("Cleaning done.")

# Filter to active users and products
print("Filtering to active users and products...")
min_user_reviews    = 5
min_product_reviews = 5
user_counts    = df["UserId"].value_counts()
product_counts = df["ProductId"].value_counts()
df_model = df[
    df["UserId"].isin(user_counts[user_counts >= min_user_reviews].index) &
    df["ProductId"].isin(product_counts[product_counts >= min_product_reviews].index)
].copy()
print(f"Filtered: {df_model.shape[0]:,} rows | {df_model['UserId'].nunique():,} users | {df_model['ProductId'].nunique():,} products")

# Define relevance (reward signal)
relevance_threshold = 4
df_model["is_relevant"] = (df_model["Score"] >= relevance_threshold).astype(int)
print(f"Relevant interaction rate: {df_model['is_relevant'].mean():.4f}")

# Train/test split
train_df, test_df = train_test_split(df_model, test_size=0.2, random_state=42)
print(f"Train: {train_df.shape[0]:,} rows | Test: {test_df.shape[0]:,} rows")
print("Data preparation complete.")

Filtered: (219540, 15) | users: 23262 | products: 17538
Relevant interaction rate: 0.7823
Train: (175632, 16) | Test: (43908, 16)


In [ ]:
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [128]:
relevance_threshold = 4
df_model["is_relevant"] = (df_model["Score"] >= relevance_threshold).astype(int)

print("Relevant interaction rate:", round(df_model["is_relevant"].mean(), 4))

Relevant interaction rate: 0.7823


In [129]:
train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    random_state=42
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train users:", train_df["UserId"].nunique())
print("Test users:", test_df["UserId"].nunique())
print("Train products:", train_df["ProductId"].nunique())
print("Test products:", test_df["ProductId"].nunique())

Train shape: (175632, 16)
Test shape: (43908, 16)
Train users: 23195
Test users: 18281
Train products: 16657
Test products: 9578


## 2. Evaluation Helper Functions

In [130]:
K = 10

print(f"Ranking cutoff K: {K}")
print(f"Relevance threshold: {relevance_threshold}")

Ranking cutoff K: 10
Relevance threshold: 4


In [131]:
def precision_at_k(eval_df, k=10, threshold=4):
    precision_scores = []

    for _, user_df in eval_df.sort_values("pred", ascending=False).groupby("UserId"):
        top_k = user_df.head(k)
        if len(top_k) == 0:
            continue
        precision_scores.append((top_k["Score"] >= threshold).mean())

    return float(np.mean(precision_scores)) if precision_scores else 0.0

In [132]:
def recall_at_k(eval_df, k=10, threshold=4):
    recall_scores = []

    for _, user_df in eval_df.sort_values("pred", ascending=False).groupby("UserId"):
        n_relevant = (user_df["Score"] >= threshold).sum()
        if n_relevant == 0:
            continue

        top_k = user_df.head(k)
        relevant_in_top_k = (top_k["Score"] >= threshold).sum()
        recall_scores.append(relevant_in_top_k / n_relevant)

    return float(np.mean(recall_scores)) if recall_scores else 0.0

In [133]:
def map_at_k(eval_df, k=10, threshold=4):
    ap_scores = []

    for _, user_df in eval_df.sort_values("pred", ascending=False).groupby("UserId"):
        n_relevant = (user_df["Score"] >= threshold).sum()
        if n_relevant == 0:
            continue

        top_k = user_df.head(k)
        relevance = (top_k["Score"] >= threshold).astype(int).values

        hits = 0
        precision_sum = 0.0
        for rank, rel in enumerate(relevance, start=1):
            if rel == 1:
                hits += 1
                precision_sum += hits / rank

        denom = min(n_relevant, k)
        ap_scores.append(precision_sum / denom if denom > 0 else 0.0)

    return float(np.mean(ap_scores)) if ap_scores else 0.0

In [134]:
def ndcg_at_k(eval_df, k=10, threshold=4):
    ndcg_scores = []

    for _, user_df in eval_df.sort_values("pred", ascending=False).groupby("UserId"):
        n_relevant = (user_df["Score"] >= threshold).sum()
        if n_relevant == 0:
            continue

        top_k = user_df.head(k)
        relevance = (top_k["Score"] >= threshold).astype(int).values

        dcg = 0.0
        for rank, rel in enumerate(relevance, start=1):
            dcg += rel / np.log2(rank + 1)

        ideal_len = min(n_relevant, k)
        idcg = 0.0
        for rank in range(1, ideal_len + 1):
            idcg += 1 / np.log2(rank + 1)

        ndcg_scores.append(dcg / idcg if idcg > 0 else 0.0)

    return float(np.mean(ndcg_scores)) if ndcg_scores else 0.0

Evaluation note: ranking metrics here are computed on each user's held-out test interactions (not on full-catalog candidate ranking). This is acceptable for lightweight offline comparison, but values can be optimistic versus a full Top-N candidate evaluation setting.


################

## **9. Hybrid Recommendations**

### 9.1 Hybrid design

### 9.2 Score combination strategy

### 9.3 Hybrid evaluation